# COVID-19 Global Data Analysis: Vaccination Dynamics
## Part 5: Rollout Velocity, Global Coverage & Mortality Decoupling

**Lead Data Engineer & Curator:** **Himanshu Bagde** ([GitHub: @Himanshubagde11](https://github.com/Himanshubagde11))  
**Primary Surveillance Source:** Our World in Data / WHO / Johns Hopkins CSSE  
**Project:** COVID-19 Global Data Analysis & Trend Visualization

This notebook evaluates:
1. Cumulative global vaccination trajectory across doses.
2. Cross-country inequality in immunization coverage.
3. Epidemiological decoupling: The mathematical relationship between vaccination rate and Case Fatality Rate (CFR).



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

DATA_PATH = ROOT_DIR / "data" / "processed" / "covid_processed.csv"
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])



### 1. Global Vaccination Timeline


In [ ]:
global_vax = df.groupby('date')[['people_vaccinated', 'people_fully_vaccinated']].sum().reset_index()
global_vax = global_vax[global_vax['date'] >= '2020-12-01']

plt.figure(figsize=(12, 6))
plt.plot(global_vax['date'], global_vax['people_vaccinated'] / 1e9, label='At Least 1 Dose (Billions)', color='#10b981', linewidth=2.2)
plt.plot(global_vax['date'], global_vax['people_fully_vaccinated'] / 1e9, label='Fully Vaccinated (Billions)', color='#059669', linestyle='--', linewidth=2.2)
plt.title("Global Cumulative Vaccine Administration Progress", fontsize=13, weight='bold')
plt.ylabel("People Vaccinated (Billions)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()



### 2. Cross-Sectional Analysis: Vaccination Coverage vs Case Fatality Rate


In [ ]:
latest = df[(df['population'] >= 2000000) & (df['total_cases'] >= 50000)].groupby('country').agg({
    'fully_vaccinated_rate': 'max',
    'case_fatality_rate': 'last',
    'population': 'max',
    'region': 'first'
}).dropna()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=latest, x='fully_vaccinated_rate', y='case_fatality_rate', hue='region', size='population', sizes=(40, 400), alpha=0.8)
sns.regplot(data=latest, x='fully_vaccinated_rate', y='case_fatality_rate', scatter=False, color='gray', line_kws={'linestyle': '--'})
plt.title("Vaccine Coverage (%) vs Case Fatality Rate (CFR %)", fontsize=13, weight='bold')
plt.xlabel("Fully Vaccinated Rate (%)")
plt.ylabel("Case Fatality Rate (%)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

